In [1]:
!rm -rf $HOME/.local/share/Trash/files

Code below will authenticate with OpenSearch endpoint using the SageMaker IAM role

In [1]:
!pip install -q boto3
!pip install -q requests
!pip install -q requests-aws4auth
!pip install -q opensearch-py
!pip install -q shapely

In [2]:
import boto3
import requests
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from requests_aws4auth import AWS4Auth
import opensearch

In [13]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import boto3

region = "ca-central-1"
aos_host = "REDACTED.ca-central-1.es.amazonaws.com"
os_secret_id = "OpenSearchSecret-geocore-semantic-search-with-opensearch-stage"

# --- Get AWS credentials ---
credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    "es",
    session_token=credentials.token,
)

# --- Create OpenSearch client with timeout + retries ---
aos_client = OpenSearch(
    hosts=[{"host": aos_host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=60,             # ⬅️ increased timeout
    max_retries=5,          # ⬅️ automatic retries
    retry_on_timeout=True   # ⬅️ retry if timeout happens
)

In [8]:
index_name = "geolocation-index"
index = {
  "settings": {
    "analysis": {
      "tokenizer": {
        "edge_ngram_tokenizer": {
          "type": "edge_ngram",
          "min_gram": 2,
          "max_gram": 20,
          "token_chars": ["letter", "digit"]
        }
      },
      "analyzer": {
        "edge_ngram_analyzer": {
          "tokenizer": "edge_ngram_tokenizer",
          "filter": ["lowercase"]
        },
        "phonetic_analyzer": {
          "tokenizer": "standard",
          "filter": ["lowercase", "my_phonetic"]
        }
      },
      "filter": {
        "my_phonetic": {
          "type": "phonetic",
          "encoder": "metaphone",
          "replace": False
        }
      },
      "normalizer": {
        "lowercase_normalizer": {
          "type": "custom",
          "filter": ["lowercase"]
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "title": {
        "type": "text",
        "analyzer": "standard",
        "fields": {
          "keyword": {
            "type": "keyword",
            "normalizer": "lowercase_normalizer"
          },
          "ngram": {
            "type": "text",
            "analyzer": "edge_ngram_analyzer",
            "search_analyzer": "standard"
          }
        }
      },
      "title_suggest": {
        "type": "completion",
        "analyzer": "phonetic_analyzer"
      },
      "street_id": { "type": "keyword" },
      "qualifier": { "type": "keyword" },
      "type": { "type": "keyword" },
      "bbox": {
        "type": "object",
        "properties": {
          "min_lon": { "type": "float" },
          "min_lat": { "type": "float" },
          "max_lon": { "type": "float" },
          "max_lat": { "type": "float" }
        }
      },
      "geometry": { "type": "geo_point" }
    }
  }
}

In [9]:
# Create or recreate the index
if aos_client.indices.exists(index=index_name):
    aos_client.indices.delete(index=index_name)
aos_client.indices.create(index=index_name, body=index,ignore=400)

{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'geolocation-index'}

In [10]:
from opensearchpy import OpenSearch, helpers
from opensearchpy.helpers import streaming_bulk
from shapely import wkt
import json
import os
import re

# --- Config ---
data_dir = "geolocation"

# --- Helper to parse "POINT (lon lat)" WKT ---
def parse_point_wkt(wkt_str):
    match = re.search(r"POINT\s*\(\s*(-?\d+\.\d+)\s+(-?\d+\.\d+)\s*\)", wkt_str)
    if match:
        lon, lat = float(match.group(1)), float(match.group(2))
        return {"lon": lon, "lat": lat}
    return None

# --- Helper to parse "POLYGON" or "MULTIPOLYGON" into bbox + centroid ---
def parse_polygon_or_multipolygon_wkt(wkt_str):
    try:
        shape = wkt.loads(wkt_str)
        minx, miny, maxx, maxy = shape.bounds
        bbox_obj = {"min_lon": minx, "min_lat": miny, "max_lon": maxx, "max_lat": maxy}
        centroid = shape.centroid
        geometry = {"lon": centroid.x, "lat": centroid.y}
        return bbox_obj, geometry
    except Exception as e:
        print(f"Geometry parse failed: {e}")
        return None, None

def prepare_actions(file_path, dataset_name, records):
    """Prepare OpenSearch bulk actions for one dataset (centroid, intersection, toponym, NTS)."""
    actions = []

    province_map = {
        "NL": "Newfoundland and Labrador",
        "PE": "Prince Edward Island",
        "NS": "Nova Scotia",
        "NB": "New Brunswick",
        "QC": "Quebec",
        "ON": "Ontario",
        "MB": "Manitoba",
        "SK": "Saskatchewan",
        "AB": "Alberta",
        "BC": "British Columbia",
        "YT": "Yukon",
        "NT": "Northwest Territories",
        "NU": "Nunavut",
        "CA": "Canada",
        "IW": "International Waters",
        "UF": "Undersea Features"
    }

    toponym_feature = {
        "AIR": "Air navigation feature",
        "BAY": "Bay",
        "BCH": "Beach",
        "CAMP": "Miscellaneous campsite",
        "CAPE": "Cape",
        "CAVE": "Cave",
        "CHAN": "Channel",
        "CITY": "City",
        "CLF": "Cliff",
        "CRAT": "Crater",
        "DMUN": "District municipality",
        "FALL": "Falls",
        "FOR": "Forest",
        "GEOG": "Geographical area",
        "GLAC": "Glacier",
        "HAM": "Hamlet",
        "HYDR": "Hydraulic construction",
        "IR": "Indian Reserve",
        "ISL": "Island",
        "LAKE": "Lake",
        "MAR": "Marine navigation feature",
        "MIL": "Military area",
        "MISC": "Miscellaneous",
        "MTN": "Mountain",
        "MUN1": "Other municipal/district area - major agglomeration",
        "MUN2": "Other municipal/district area - miscellaneous",
        "PARK": "Conservation area",
        "PLN": "Plain",
        "PROV": "Province",
        "RAIL": "Railway feature",
        "RAP": "Rapids",
        "RECR": "Recreational site",
        "RES": "Natural resources site",
        "RIV": "River",
        "RIVF": "River feature",
        "ROAD": "Road feature",
        "SEA": "Sea",
        "SEAF": "Sea feature",
        "SEAU": "Undersea feature",
        "SHL": "Shoal",
        "SITE": "Miscellaneous site",
        "SPRG": "Spring",
        "TERR": "Territory",
        "TOWN": "Town",
        "UNP": "Unincorporated area",
        "VALL": "Valley",
        "VEGL": "Low vegetation",
        "VILG": "Village"
    }

    for i, item in enumerate(records):
        # --- Detect geometry field ---
        geom_wkt = item.get("geom") or item.get("shape")
        if not geom_wkt:
            continue

        dataset_lower = dataset_name.lower()

        # === Handle NTS datasets ===
        if "nts" in dataset_lower:
            tile_id = item.get("tile_id")
            scale = str(item.get("scale_denom") or "")
            title = item.get("tile_name") or tile_id or "(unnamed tile)"
            title = f"{tile_id} {title}"
            doc_id = f"{tile_id}_{scale}" if tile_id else f"nts_{os.path.basename(file_path)}_{i}"

            bbox, geometry = parse_polygon_or_multipolygon_wkt(geom_wkt)
            if not geometry:
                continue

            actions.append({
                "_index": index_name,
                "_id": doc_id,
                "_source": {
                    "title": title,
                    "title_suggest": {    "input": [
                                                title.split(",")[0]           # only main name
                                            ],
                                           "weight": 2},
                    "scale": scale,
                    "qualifier": "LOCATION",
                    "type": "ca.gc.nrcan.geoloc.data.model.NTS",
                    "bbox": bbox,
                    "geometry": geometry
                }
            })
            continue

        # === Handle postal code datasets ===
        if "postal_code" in dataset_lower:
            code = item.get("code") or item.get("fsa")
            if not code:
                continue
            geometry = parse_point_wkt(geom_wkt)
            if not geometry:
                continue

            title = code
            doc_id = item.get("bdg_id") or f"postal_{code}_{i}"

            actions.append({
                "_index": index_name,
                "_id": doc_id,
                "_source": {
                    "title": title,
                    "title_suggest": {    "input": [
                                                title.split(",")[0]           # only main name
                                            ],
                                          "weight": 2},
                    "qualifier": "INTERPOLATED_CENTROID",
                    "type": "ca.gc.nrcan.geoloc.data.model.PostalCode",
                    "geometry": geometry
                }
            })
            continue

        # === Handle non-NTS datasets ===
        if "POINT" in geom_wkt:
            geometry = parse_point_wkt(geom_wkt)
            bbox = None
        elif "POLYGON" in geom_wkt:
            bbox, geometry = parse_polygon_or_multipolygon_wkt(geom_wkt)
        else:
            continue

        if not geometry:
            continue

        place = item.get("official_place_name")# or item.get("location")
        province_code = item.get("political_division_abb")# or item.get("prov_terr_list")
        province = province_map.get(province_code, province_code)
        name = item.get("name")
        qualifier = "LOCATION"
        street_id = ""
        dataset = ""

        if "centroide" in dataset_lower:
            street = item.get("official_street_name")
            street_id = item.get("street_id")
            parts = [p for p in [street, place, province] if p]
            title = ", ".join(parts)
            qualifier = "INTERPOLATED_CENTROID"
            dataset = "ca.gc.nrcan.geoloc.data.model.Street"
        elif "intersection" in dataset_lower:
            street = item.get("official_street_name_search").replace(",", " & ")
            parts = [p for p in [street, place, province] if p]
            title = ", ".join(parts)
            qualifier = "LOCATION"
            dataset = "ca.gc.nrcan.geoloc.data.model.Intersection"
        elif name: #toponym
            location = item.get("location")
            province_code = item.get("prov_terr_list")
            province = province_map.get(province_code, province_code)
            concise_code = item.get("concise_code")
            feature = toponym_feature.get(concise_code, concise_code)

            parts = [p for p in [name, location, province] if p]
            title = ", ".join(parts)

            if feature:
                title = f"{title} ({feature})"

            dataset = "ca.gc.nrcan.geoloc.data.model.Geoname"
            qualifier = "LOCATION"
        else:
            title = "(untitled)"

        doc_id = (
            item.get("feature_id")
            or item.get("bdg_id")
            or item.get("toponymic_feature_id")
        )

        if dataset == "ca.gc.nrcan.geoloc.data.model.Geoname":
            suggestion_weight = 3
        else:
            suggestion_weight = 1

        actions.append({
            "_index": index_name,
            "_id": doc_id,
            "_source": {
                "title": title,
                "title_suggest": {    "input": [
                                            title.split(",")[0]           # only main name
                                        ],
                                      "weight": suggestion_weight},
                "street_id": street_id,
                "qualifier": qualifier,
                "type": dataset,
                "bbox": bbox,
                "geometry": geometry
            }
        })

    return actions


# --- JSON files to process ---
json_files = [
    "road_segment_intersection_0_202511071006.json",
    "toponym_2_202511070939.json",
    "road_segment_centroide_0_202511071047.json",
    "nts_all_scales_2_202511070934.json",
    "postal_code_centroid_0_202511070905.json"
]

all_actions = []

for file_name in json_files:
    file_path = os.path.join(data_dir, file_name)
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = json.load(f)
    except json.JSONDecodeError as e:
        print(f"Skipping {file_name}: invalid JSON ({e})")
        continue

    dataset_name = next(iter(content))
    records = content[dataset_name]

    print(f"Processing {file_name} ({len(records)} records)...")
    actions = prepare_actions(file_path, dataset_name, records)
    all_actions.extend(actions)

# --- Bulk index all documents ---
if all_actions:
    indexed = 0
    print(f"📌 Starting streaming bulk upload of {len(all_actions)} docs...")

    for ok, result in streaming_bulk(
        aos_client,
        all_actions,
        chunk_size=350,         # adjust if too slow; try 200–500
        max_retries=5,
        yield_ok=True,
        request_timeout=180
    ):
        if not ok:
            print("❌ Failed doc:", result)
        else:
            indexed += 1
            # Print progress every 2000 docs (you can adjust)
            if indexed % 5000 == 0:
                print(f"  → {indexed} documents indexed...")

    print(f"🎉 Done! Indexed {indexed} documents into '{index_name}'.")
else:
    print("⚠️ No records were indexed.")

Processing road_segment_intersection_0_202511071006.json (866474 records)...
Processing toponym_2_202511070939.json (364756 records)...
Processing road_segment_centroide_0_202511071047.json (857180 records)...
Processing nts_all_scales_2_202511070934.json (20390 records)...
Processing postal_code_centroid_0_202511070905.json (1636 records)...
📌 Starting streaming bulk upload of 2110432 docs...
  → 5000 documents indexed...
  → 10000 documents indexed...
  → 15000 documents indexed...
  → 20000 documents indexed...
  → 25000 documents indexed...
  → 30000 documents indexed...
  → 35000 documents indexed...
  → 40000 documents indexed...
  → 45000 documents indexed...
  → 50000 documents indexed...
  → 55000 documents indexed...
  → 60000 documents indexed...
  → 65000 documents indexed...
  → 70000 documents indexed...
  → 75000 documents indexed...
  → 80000 documents indexed...
  → 85000 documents indexed...
  → 90000 documents indexed...
  → 95000 documents indexed...
  → 100000 docu

In [11]:
index_name_address_range = "geolocation-index-address-range"
address_range_mapping = {
    "mappings": {
        "properties": {
            "bdg_id": {"type": "keyword"},
            "feature_id": {"type": "keyword"},
            "street_id": {"type": "keyword"},
            "zt_id": {"type": "keyword"},
            "min_house_number": {"type": "integer"},
            "max_house_number": {"type": "integer"},
            "digitizing_direction": {"type": "integer"},
            "numbering_method": {"type": "integer"},
            "geometry": {
                "type": "geo_shape"
            }
        }
    }
}

In [14]:
# Create or recreate the index
if aos_client.indices.exists(index=index_name_address_range):
    aos_client.indices.delete(index=index_name_address_range)
aos_client.indices.create(index=index_name_address_range, body=address_range_mapping,ignore=400)

{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'geolocation-index-address-range'}

In [15]:
from opensearchpy import OpenSearch, helpers
import json
import os
import re

# --- Config ---
data_dir = "geolocation"

index_name_address_range = "geolocation-index-address-range"
address_range_mapping = {
    "mappings": {
        "properties": {
            "bdg_id": {"type": "keyword"},
            "feature_id": {"type": "keyword"},
            "street_id": {"type": "keyword"},
            "zt_id": {"type": "keyword"},
            "min_house_number": {"type": "integer"},
            "max_house_number": {"type": "integer"},
            "digitizing_direction": {"type": "integer"},
            "numbering_method": {"type": "integer"},
            "geometry": {
                "type": "geo_shape"
            }
        }
    }
}

# Create or recreate index
if aos_client.indices.exists(index=index_name_address_range):
    aos_client.indices.delete(index=index_name_address_range)
aos_client.indices.create(index=index_name_address_range, body=address_range_mapping, ignore=400)


def safe_int(v):
    try:
        return int(v)
    except:
        return None


# --- WKT to GeoJSON (fast) ---
def linestring_wkt_to_geojson(wkt: str):
    text = wkt[11:].strip()         # remove "LINESTRING"
    text = text.lstrip("(").rstrip(")")
    coords = []
    for pair in text.split(","):
        lon, lat = pair.strip().split()
        coords.append([float(lon), float(lat)])
    return {"type": "LineString", "coordinates": coords}


# --- BULK INDEXING ---
def index_address_ranges(json_data, index_name):
    actions = []
    count = 0

    for rec in json_data["address_range_1"]:
        geom = rec.get("geom")
        if not geom or not geom.startswith("LINESTRING"):
            continue

        geojson = linestring_wkt_to_geojson(geom)

        doc = {
            "_index": index_name,
            "_source": {
                "bdg_id": rec.get("bdg_id"),
                "feature_id": rec.get("feature_id"),
                "street_id": rec.get("street_id"),
                "zt_id": rec.get("zt_id"),
                "min_house_number": safe_int(rec.get("min_house_number")),
                "max_house_number": safe_int(rec.get("max_house_number")),
                "digitizing_direction": rec.get("digitizing_direction"),
                "numbering_method": rec.get("numbering_method"),
                "geometry": geojson
            }
        }

        actions.append(doc)
        count += 1

        # Bulk index in chunks of 1000
        if len(actions) >= 1000:
            helpers.bulk(aos_client, actions)
            actions = []
            #print(f"Indexed {count} records...")

    # Final flush
    if actions:
        helpers.bulk(aos_client, actions)
        #print(f"Indexed final {len(actions)} records")

    print(f"✅ Done. Indexed {count} address ranges.")


# MAIN
json_files = ["address_range_1_202511071133.json",
              "address_range_1_202511071133_2.json",
              "address_range_1_202511071133_3.json",
              "address_range_1_202511071133_4.json",
              "address_range_1_202511071133_5.json"
             ]

for file_name in json_files:
    file_path = os.path.join(data_dir, file_name)
    print(f"📂 Reading {file_path}...")

    with open(file_path, "r", encoding="utf-8") as f:
        content = json.load(f)

    print(f"📍 Indexing {len(content['address_range_1'])} address ranges...")

    index_address_ranges(content, index_name_address_range)

📂 Reading geolocation/address_range_1_202511071133.json...
📍 Indexing 434921 address ranges...
✅ Done. Indexed 434921 address ranges.
📂 Reading geolocation/address_range_1_202511071133_2.json...
📍 Indexing 470913 address ranges...
✅ Done. Indexed 470913 address ranges.
📂 Reading geolocation/address_range_1_202511071133_3.json...
📍 Indexing 455192 address ranges...
✅ Done. Indexed 455192 address ranges.
📂 Reading geolocation/address_range_1_202511071133_4.json...
📍 Indexing 327064 address ranges...
✅ Done. Indexed 327064 address ranges.
📂 Reading geolocation/address_range_1_202511071133_5.json...
📍 Indexing 272393 address ranges...
✅ Done. Indexed 272393 address ranges.
